# Tarefa 2

## 1. Cite 5 diferenças entre o AdaBoost(Adaptive Boosting) e o GBM(Gradient Boosting Machine).

- **Forma de Atualização dos Pesos dos Exemplos**: O AdaBoost atualiza os pesos das amostras com base nos erros do modelo anterior. As amostras mal classificadas ganham mais peso na próxima iteração. O GBM ajusta o modelo minimizando diretamente uma função de perda (ex: erro quadrático, log loss), utilizando o gradiente dessa função. Não há manipulação explícita de pesos das amostras.

- **Função de Perda**: O AdaBoost usa implicitamente a função de perda exponencial. O GBM permite o uso de diversas funções de perda, como erro quadrático, log loss, Huber, entre outras — o que o torna mais flexível.

- **Tipo de Base Learner (Modelo Base)**: O AdaBoost tradicionalmente usa árvores de decisão muito rasas (stumps) — árvores com apenas um nível. O GBM normalmente usa árvores mais profundas, com profundidade ajustável, o que permite maior capacidade de modelagem.

- **Sensibilidade a Outliers**: O AdaBoost é muito sensível a outliers, pois o peso dos erros cresce exponencialmente, e outliers acabam influenciando fortemente o modelo. O GBM é menos sensível a outliers (dependendo da função de perda escolhida, como Huber), pois pode suavizar os impactos usando gradientes.

- **Velocidade de Convergência e Estabilidade**: O AdaBoost pode convergir mais rápido em alguns cenários simples, mas é menos estável quando os dados são ruidosos. O GBM geralmente converge de forma mais robusta e estável, com maior controle (por exemplo, através do learning rate e da profundidade das árvores).

## 2. Acesse o link Scikit-learn - GBM, leia a explicação (traduza se for preciso) e crie um jupyter notebook contendo o exemplo de classificação e de regressão do GBM.

In [5]:
from sklearn.datasets import make_hastie_10_2
from sklearn.ensemble import GradientBoostingClassifier

X, y = make_hastie_10_2(random_state=0)
X_train, X_test = X[:2000], X[2000:]
y_train, y_test = y[:2000], y[2000:]

clf = GradientBoostingClassifier(
    n_estimators=100, 
    learning_rate=1.0,
    max_depth=1, 
    random_state=0
).fit(X_train, y_train)

clf.score(X_test, y_test)

0.913

In [6]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_friedman1
from sklearn.ensemble import GradientBoostingRegressor

X, y = make_friedman1(n_samples=1200, random_state=0, noise=1.0)
X_train, X_test = X[:200], X[200:]
y_train, y_test = y[:200], y[200:]

est = GradientBoostingRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=1, 
    random_state=0,
    loss='squared_error'
).fit(X_train, y_train)

mean_squared_error(y_test, est.predict(X_test))

5.009154859960321

## 3. Cite 5 Hyperparametros importantes no GBM.


- **n_estimators**: Número de árvores (estimadores) na sequência do boosting. Mais árvores geralmente melhoram a performance, mas aumentam o tempo de treinamento e o risco de overfitting.

- **learning_rate**: Taxa de aprendizado; determina o quanto cada árvore contribui para o modelo final. Taxas menores tornam o modelo mais robusto, mas exigem mais árvores (n_estimators) para alcançar boa performance.

- **max_depth**: Profundidade máxima de cada árvore individual. Controla a complexidade do modelo. Árvores mais profundas aprendem padrões complexos, mas podem causar overfitting.

- **subsample**: Fração das amostras usadas para treinar cada árvore (amostragem sem reposição). Ajuda a reduzir o overfitting e melhora a generalização. Um valor < 1.0 ativa um comportamento similar ao Stochastic Gradient Boosting.

- **min_samples_split**: Número mínimo de amostras necessárias para dividir um nó interno da árvore. Controla o crescimento das árvores. Valores maiores resultam em árvores mais rasas e generalizadas.

## 4. Utilize o GridSearch para encontrar os melhores hyperparametros para o conjunto de dados do exemplo

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_hastie_10_2
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

X, y = make_hastie_10_2(random_state=0)
X_train, X_test = X[:2000], X[2000:]
y_train, y_test = y[:2000], y[2000:]

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5, 1.0],
    'max_depth': [1, 2, 3],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=0),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Melhores hiperparâmetros:", grid_search.best_params_)
print(f"Acurácia no teste: {accuracy_score(y_test, y_pred):.4f}")


Fitting 5 folds for each of 72 candidates, totalling 360 fits
Melhores hiperparâmetros: {'learning_rate': 1.0, 'max_depth': 1, 'n_estimators': 200, 'subsample': 1.0}
Acurácia no teste: 0.9305


In [11]:
from sklearn.datasets import make_friedman1
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Dados de regressão
X, y = make_friedman1(n_samples=1200, random_state=0, noise=1.0)
X_train, X_test = X[:200], X[200:]
y_train, y_test = y[:200], y[200:]

# Grade de hiperparâmetros
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [1, 2, 3],
    'subsample': [0.8, 1.0],
    'loss': ['squared_error']
}

# GridSearch para regressão
grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=0),
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Melhores hiperparâmetros:", grid_search.best_params_)
print(f"MSE no teste: {mean_squared_error(y_test, y_pred):.4f}")
print(f"R² no teste: {r2_score(y_test, y_pred):.4f}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Melhores hiperparâmetros: {'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 2, 'n_estimators': 200, 'subsample': 0.8}
MSE no teste: 3.4166
R² no teste: 0.8676


## 5. Acessando o artigo do Jerome Friedman (Stochastic) e pensando no nome dado ao Stochastic GBM, qual é a maior diferença entre os dois algoritmos?

A principal diferença entre o GBM tradicional e o Stochastic GBM é que o Stochastic usa apenas uma parte aleatória dos dados para treinar cada árvore, enquanto o tradicional usa todos os dados. Isso ajuda a evitar overfitting, deixa o modelo mais robusto e pode acelerar o treinamento.
